# ✈️ AeroLex — Building a Production-Grade RAG Pipeline

A hands-on build of a multi-stage Retrieval-Augmented Generation system over
**airline passenger-rights law** (EU 261/2004, US DOT 14 CFR 260, Delta Contracts
of Carriage).

**The pipeline:**
```
question
  └─▶ condense follow-up            (conversational memory)       [#8]
  └─▶ BM25 lexical  +  Gemini dense  (hybrid retrieval)           [#2,#3,#4]
  └─▶ reciprocal rank fusion         (merge the two)              [#5]
  └─▶ FLAIR feedback re-scoring      (learn from 👍/👎)            [#10]
  └─▶ cross-encoder re-ranking       (precise top-k)              [#6]
  └─▶ bounded reference expansion  (supporting evidence)        [#1]
  └─▶ grounded answer + [[n]] cites  (+ long-term profile)        [#7,#9]
```

**How to run:** pick the **`Python (aerolex RAG)`** kernel, then run top to bottom.
Feature #1 runs offline. Later stages need matching embeddings and installed
retrieval models; Gemini embedding/generation needs a `GEMINI_API_KEY`. You built each stage? The final section assembles them with the
`aerolex` library.


# Why and when do we use RAG?
# Best practices
# See if we can implement SOTA for our respective sections - if yes, great, if no, try to do a simpler version or talk about it

## 0 · Setup

In [ ]:
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'aerolex').is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))
import time
from pathlib import Path
import numpy as np
from aerolex import config
from aerolex.loaders import load_documents
from aerolex.chunking import ChunkCacheError, chunk_corpus, load_chunks, save_chunks
from aerolex.context import ReferenceExpander
from aerolex.store import InMemoryVectorStore
from aerolex.embeddings import GeminiEmbedder
from aerolex.bm25 import BM25Index

In [ ]:
# Set API key, LLM, Embedding Model 
EMBED_MODEL = "gemini-embedding-001"
EMBED_DIM = 768
GEN_MODEL = "gemini-3.5-flash-lite"
KEY = config.load_api_key()
CACHE_DIR = config.CACHE_DIR

## 1 · Reference-aware Hierarchical Chunking · Feature #1 - Advitya

Legal text has a hierarchy: **document -> Article / section / RULE -> paragraph**.
Keep these boundaries, pack paragraphs/legal blocks into **at most 320 words**,
and fall back to sentences and then word windows for oversized blocks.
Up to **40 words of overlap are included in that budget**, never across sections.
These are word budgets, not exact model-token counts.

This is **structure-first chunking**, not an embedding-similarity semantic splitter.
Word/paragraph splitting is not universally better: evaluate the evidence needed
for your questions. Crucially, better boundaries alone cannot fetch another article.

```text
Baseline:        hybrid search -> rerank -> top-k passages -> answer
Reference-aware: hybrid search -> rerank -> seed sections
                 -> bounded outgoing + incoming references -> evidence -> answer
```

Each chunk's JSON has `references` (target chunk IDs) and `reference_mentions`
(verbatim mention, subclause locator, scope, and resolution status).
`Article 9(1)(b)` resolves to all parts of Article 9; it is not an exact subclause index.
References to other instruments stay unresolved unless explicitly mapped to this corpus.

**Why both directions?** Article 5 invokes Articles 7–9. Article 7 contains amounts
but does not point back to Article 5's exceptions. Incoming links discover invoking
provisions; they do not prove those provisions apply to the question.

The backend owns legal parsing (`references.py`). The reusable template below
only chunks documents, follows IDs, and assembles evidence (`context.py`).
Defaults: one hop, two incoming source sections per seed, 20 added chunks,
6,000 evidence words. Whole sections that do not fit are reported as omitted.
Complete seed sections that cannot fit raise an error instead of losing conditions.


In [ ]:
docs = load_documents()
for d in docs:
    print(f"{d.doc_id:16s} {d.doc_type:6s} {len(d.text):>7,d} chars  {d.title}")

try:
    chunks = load_chunks(documents=docs)
except (FileNotFoundError, ChunkCacheError) as error:
    print(f"Building current chunk cache: {error}")
    chunks = chunk_corpus(docs)
    save_chunks(chunks, documents=docs)

by_id = {c.chunk_id: c for c in chunks}
expander = ReferenceExpander(chunks)
art5 = next(c for c in chunks if c.section_id == ("eu261", "Article 5"))
art7 = next(c for c in chunks if c.section_id == ("eu261", "Article 7"))
assert all(len(c.text.split()) <= config.CHUNK_MAX_WORDS for c in chunks)
print(f"Loaded {len(chunks)} chunks; largest: {max(len(c.text.split()) for c in chunks)} words")


In [ ]:
import json
from dataclasses import asdict

print(json.dumps(asdict(art5), ensure_ascii=False, indent=2))
print("Resolved targets:", list(dict.fromkeys(by_id[cid].citation for cid in art5.references)))
external = next(m for c in chunks if c.section_id == ("eu261", "Article 2")
                for m in c.reference_mentions if m.status == "external")
print("Not guessed as local Article 2:", json.dumps(asdict(external), ensure_ascii=False))


### Two small, reusable exercises

1. Implement `fetch_referenced_chunks`: follow `chunk.references` through `by_id`,
   deduplicate IDs, and return the referenced chunks.
2. Implement `assemble_context`: call the shared expander on the selected seeds.
   Inspect its `chunks`, `provenance`, and `diagnostics`.

The solution notebook contains the completed functions. The demonstration after
these exercises uses the backend directly so Feature #1 can run offline even
before you fill in the functions.


In [ ]:
def fetch_referenced_chunks(chunk, by_id):
    # TODO: follow and deduplicate the target IDs.
    raise NotImplementedError("Fill in the reference-ID lookup")

def assemble_context(seed_chunks):
    # TODO: call the reusable expander.
    raise NotImplementedError("Call the shared reference expander")

# After implementing, try fetch_referenced_chunks(art5, by_id)
# and assemble_context([art7]).metadata()


In [ ]:
for seed in (art5, art7):
    context = expander.expand([seed])
    print("\nBaseline:", seed.citation)
    print("Expanded sections:", list(dict.fromkeys(c.section_id for c in context.chunks)))
    print("Discovery edges:", list(dict.fromkeys(
        (e.direction, e.source_section, e.target_section) for e in context.provenance)))
    print("Limits / missing evidence:", json.dumps(context.metadata()["diagnostics"], ensure_ascii=False))
    print("Evidence words:", context.metadata()["evidence_words"])

# Seed-controlled evidence coverage, NOT a claim about search or answer accuracy.
examples = [
    ("Cancellation compensation", art5, {"Article 5", "Article 7"}),
    ("Rerouting reduction", art7, {"Article 7", "Article 8"}),
    ("Care after cancellation", art5, {"Article 5", "Article 9"}),
]
print("\nQuestion | baseline coverage | expanded coverage | words")
for question, seed, required in examples:
    evidence = expander.expand([seed])
    present = {c.section_label for c in evidence.chunks}
    print(question, f"{len(required & {seed.section_label})}/{len(required)}",
          f"{len(required & present)}/{len(required)}", evidence.metadata()["evidence_words"], sep=" | ")


**Inspect before answering.** Try `expander.expand([art7], max_added_chunks=0)`
to see budget omissions, or `max_hops=2` to explore an additional reference hop.
Unresolved citations and budget/depth limits remain visible; no partial target
section is silently presented as complete.

Graph coverage is not legal correctness. Applicability may require provisions not
explicitly linked to a seed, facts missing from the question, or law outside these
PDFs. In particular, the EU261 PDF alone does not establish every later delay-
compensation judgment. We will expand evidence **after reranking** below, without
pruning the referenced conditions back down to the original top-k.


## Emebdding Creation, Chunking eval - Nutan

In [ ]:
print(f"{len(chunks)} chunks loaded!")

In [ ]:
if not KEY:
    print("GEMINI_API_KEY not set.")

print(f"Embedding {len(chunks)} chunks with {EMBED_MODEL} "
        f"(dim={EMBED_DIM}) …")

t0 = time.time()
embedder = GeminiEmbedder(api_key=KEY, model=EMBED_MODEL, dim=EMBED_DIM)
matrix = embedder.embed_documents([c.searchable_text for c in chunks])
print(f"Embedded in {time.time() - t0:.1f}s, shape={matrix.shape}")

In [ ]:
# create a matrix with id and store using numpy

store = InMemoryVectorStore.from_chunks(chunks, matrix)
store.save(Path(CACHE_DIR) / "new_embeddings.npz")

In [ ]:
# if time out errors load, already created vector store
store = InMemoryVectorStore.load(Path(CACHE_DIR) / "new_embeddings.npz")
store.validate_chunks(chunks)
print (f"{store.matrix.shape[0]} Embeddings loaded successfully of shape {store.matrix.shape[1]}!")

## 2 · Lexical Search with BM25  ·  Feature #2 - Nutan

BM25 (Best Match 25) is a probabilistic ranking algorithm to measure how relevant a document is to a query, combining term frequency, inverse document frequency, and document‑length normalization.
Use `BM25Okapi` directly with the same stopword-filtering tokenizer for chunks and queries. Scores are lexical relevance scores, not cosine similarities.

### Keyword tokenization

In [6]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def tokenize(text: str) -> list[str]:
    # TODO Fetch list of keywords 
    tokens = re.findall(r"[a-z]+|\u00A7|\d+(?:[./-]\d+)*", text.lower())
    return [token for token in tokens if token not in ENGLISH_STOP_WORDS]

assert tokenize("What is the refund for a cancelled flight?") == ["refund", "cancelled", "flight"]
print("tokenize OK")

tokenize OK


In [7]:
# Tokenize the query for BM25 search
query_tokens = tokenize("What is the refund for a cancelled flight")

In [8]:
# Initialize BM25 index with chunk IDs and their corresponding searchable text
bm25 = BM25Index([c.chunk_id for c in chunks], [c.searchable_text for c in chunks])

In [ ]:
kw_search_chunks = []
for cid, score in bm25.search(query_tokens, top_k=5):
    print(f"{score:6.2f}  {by_id[cid].citation}")
    kw_search_chunks.append((cid, score))

## 3 · Vector Search - Nutan

Gemini embeds every chunk into a vector; the whole corpus is saved in one NumPy matrix.
Similarity is just a **normalized dot product** (cosine) — no vector database.

### 🛠️ TODO: write `cosine_search` (normalized dot product top-k)

In [12]:
def cosine_search(query_vec, matrix, ids, top_k=5):

    # TODO: 1) L2-normalize query_vec and each row of matrix
    q_hat = query_vec / np.sqrt(np.sum(query_vec ** 2))
    matrix_hat = matrix / np.sqrt(np.sum(matrix ** 2, axis=1, keepdims=True))
    #       2) scores = matrix @ query   (cosine, since normalized)
    scores = matrix_hat @ q_hat
    #       3) return the top_k (id, score) pairs, highest first
    top_k = np.argsort(-scores)[:top_k]
    top_chunks = [(ids[i], scores[i]) for i in top_k]
    return top_chunks

In [13]:
# def cosine_search(query_vec, matrix, ids, top_k=5):
    # TODO: 1) L2-normalize query_vec and each row of matrix
    #       2) scores = matrix @ query   (cosine, since normalized)
    #       3) return the top_k (id, score) pairs, highest first
    # raise NotImplementedError("your turn!")

_hit = cosine_search(store.matrix[0], store.matrix, store.ids, top_k=5)[0]
assert _hit[0] == store.ids[0] and abs(_hit[1] - 1.0) < 1e-4
print("cosine_search OK")

cosine_search OK


In [ ]:
# Embed a real query (1 API call) and search. Needs a key; else we reuse a chunk vector.
query = "What is the refund for a cancelled flight?"

query_vector = GeminiEmbedder(api_key=KEY, model=EMBED_MODEL, dim=EMBED_DIM).embed_query(query)


dense_hits = cosine_search(query_vector, store.matrix, store.ids, top_k=5)
vector_search_chunks = []
for cid, s in dense_hits:
    vector_search_chunks.append((cid, s))
    print(f"{s:5.3f}  {by_id[cid].citation}")

## 4 · Reciprocal Rank Fusion  ·  Feature #5

BM25 scores and cosine similarity scores live on different scales and produce two different sets of ranked documents. So we fuse by **rank**:
each list contributes `1 / (k + rank)` to an id's score (`k = 60`).

### 🛠️ TODO: write `rrf`

In [ ]:
vector_search_chunks

In [ ]:
kw_search_chunks

In [68]:
def rrf(vector_search_chunks, kw_search_chunks, k=60):
    # TODO 
    combined_chunks = [chunk[0] for chunk in kw_search_chunks] + [chunk[0] for chunk in vector_search_chunks]
    from collections import defaultdict
    kw_chunk_rank = defaultdict(int, {chunk[0]: index + 1 for index, chunk in enumerate(kw_search_chunks)})
    vector_chunk_rank = defaultdict(int, {chunk[0]: index + 1 for index, chunk in enumerate(vector_search_chunks)})
    rrf_scores = {}
    for chunk in combined_chunks:
        r1 = kw_chunk_rank[chunk] # rank in bm25
        r2 = vector_chunk_rank[chunk] # rank in vector search
        bm25_term = 1/(r1 + k) if r1 > 0 else 0
        vector_term = 1/(r2 + k) if r2 > 0 else 0
        final_rrf_score = bm25_term + vector_term
        rrf_scores[chunk] = final_rrf_score
    ranked_chunks = sorted(
    rrf_scores.items(), key=lambda item: item[1], reverse=True
)
    return ranked_chunks

In [ ]:
ranked_chunks = rrf(kw_search_chunks, vector_search_chunks, k=60)

In [ ]:
ranked_chunks

In [ ]:
by_id[ranked_chunks[0][0]]

In [ ]:
# More efficient way
# def rrf(rankings, k=60, top_k=10):
#     scores = {}
#     for ranking in rankings:
#         for rank, chunk_id in enumerate(ranking, start=1):
#             scores[chunk_id] = scores.get(chunk_id, 0.0) + 1.0 / (k + rank)
#     return sorted(scores.items(), key=lambda item: item[1], reverse=True)[:top_k]

# assert rrf([["b", "a", "c"], ["b", "a", "d"]])[0][0] == "b"
# print("rrf OK")

rrf OK


## 5 · Cross-Encoder Re-Ranking  ·  Feature #6 - TBD

First-stage retrieval scores query and document *independently*. A cross-encoder
reads them **together** (deep cross-attention) and is far more precise — so we run
it only over the small fused pool. (Model is pre-cached; runs locally.)

In [ ]:
from aerolex.rerank import CrossEncoderReranker

candidates = [(cid, by_id[cid].searchable_text) for cid, _ in ranked_chunks]
reranked = CrossEncoderReranker().rerank(query, candidates, top_k=config.RERANK_TOP_K)
print("Re-ranked top chunks:")
for cid, s in reranked:
    print(f"{s:7.2f}  {by_id[cid].citation}")
# Reranking selects seeds; reference expansion assembles supporting evidence.
baseline_chunks = [by_id[cid] for cid, _ in reranked]
context = expander.expand(baseline_chunks)
final_chunks = context.chunks
print("Seed chunks:", len(baseline_chunks), "| evidence chunks:", len(final_chunks))
print("Evidence limits:", context.metadata()["diagnostics"])

In [ ]:
from aerolex.generate import Generator

generator = Generator(api_key=KEY, model=GEN_MODEL)
baseline_result = generator.generate(query, baseline_chunks)
print("BASELINE (top-k only):\n", baseline_result["answer"])


In [ ]:
result = generator.generate(query, final_chunks, context_diagnostics=context.metadata())
print("REFERENCE-AWARE:\n", result["answer"])
print("Evidence notes:", result["context_expansion"]["diagnostics"])


## 6 · Grounded Answer with Inline Citations  ·  Feature #7 - Advitya

- What is the state of the art in citations?
- See if we can implement it 

The model answers **only** from numbered passages and tags each claim with `[[n]]`.
A regex post-processor then validates those tags against the real passages and
**drops hallucinated ones** — the difference between "looks cited" and "is cited".

### 🛠️ TODO: write `resolve_citations`

In [ ]:
CITE = re.compile(r"\[\[(\d+)\]\]")

def resolve_citations(answer, index):
    # TODO: keep only [[n]] whose n is a key in `index`; delete the rest.
    # Return (cleaned_answer, ["[[n]] <citation>", ...] for the n's actually used).
    raise NotImplementedError("your turn!")

_c, _r = resolve_citations("A [[1]] and B [[9]].", {1: "EU261 Art 7", 2: "EU261 Art 5"})
assert "[[9]]" not in _c and _r == ["[[1]] EU261 Art 7"]
print("resolve_citations OK")

In [ ]:
# Reuse the reference-aware result above; avoid another generation call.
print("OFFLINE" if result["offline"] else "GEMINI", "answer:\n")
print(result["answer"])
print("\nReferences:")
for reference in result["references"]:
    print("  ", reference)


## Evaluation Strategies - Nutan

## 10 · The Full Pipeline

Assemble the same library components, including reference expansion after seed
reranking. Set `expand_references=False` for the original top-k baseline.
This example does not require the solution notebook's memory/profile cells.


In [ ]:
from aerolex.pipeline import RagPipeline

if KEY:
    pipe = RagPipeline(chunks, store, bm25,
                       GeminiEmbedder(api_key=KEY, model=EMBED_MODEL, dim=EMBED_DIM),
                       reranker=CrossEncoderReranker(), generator=generator)
    out = pipe.answer('For an EU261-covered non-intra-EU 4000 km flight cancelled one day before departure, with no extraordinary circumstances, I accept rerouting that arrives three hours late. What compensation amount and care apply?', expand_references=True)
    print(out["answer"])
    print("References:", out["references"])
    print("Evidence notes:", out["context_expansion"]["diagnostics"])
else:
    print("Set GEMINI_API_KEY to run the full generative pipeline; Feature #1 is offline.")


## 11 · Where to go next — citations, state of the art

What we built is **Level 1** citation: the model self-tags `[[n]]`, and we validate
the *number is real*. State-of-the-art adds **verification**:

- **Level 2** structured output — model returns `{sentence, support_ids, quote}`.
- **Level 3** quote-grounded spans — match the quote back into the source for exact
  offsets; drop unverifiable quotes.
- **Level 4** NLI / entailment (ALCE) — check the passage actually *entails* the
  claim; score citation **precision & recall**.

The retrieval stack (chunking → hybrid → fusion → re-rank) is where most answer
quality is won. 🎓 You built all of it.